In [ ]:
# --- imports ---
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# SETTINGS - edit these values as needed
# Keep the anchor symbol last in the list.
# Use ["long"], ["short"], or ["long", "short"].
# ============================================================
sorted_symbols_list = ["CWB", "ICVT"]
trade_directions = ["long", "short"]
moving_avg_days = 20
start_date = pd.Timestamp("2026-01-02")
end_date = pd.Timestamp("2026-07-31")
hurdle_step = 0.001
number_of_entry_steps = 10
input_directory = Path("with_unit_prices")
output_directory = Path("with_positions")


def build_hurdles(direction):
    sign = -1 if direction == "long" else 1
    entry_hurdles = [0.0] + [
        round(sign * hurdle_step * i, 4)
        for i in range(1, number_of_entry_steps + 1)
    ]
    return entry_hurdles


def add_positions(base_df, direction, entry_hurdle, exit_hurdle):
    df = base_df.copy()
    entry_column = f"go {direction}"
    exit_column = f"exit {direction}"

    if direction == "long":
        df[entry_column] = df["unit price pct diff"] < entry_hurdle
        df[exit_column] = df["unit price pct diff"] > exit_hurdle
        position_value = 1
    else:
        df[entry_column] = df["unit price pct diff"] > entry_hurdle
        df[exit_column] = df["unit price pct diff"] < exit_hurdle
        position_value = -1

    df["current_position"] = 0
    df["new_position"] = 0
    current_position_column = df.columns.get_loc("current_position")
    new_position_column = df.columns.get_loc("new_position")

    for row_index in range(moving_avg_days, len(df)):
        current_position = df.iat[row_index - 1, new_position_column]
        new_position = current_position

        if current_position == 0 and df[entry_column].iat[row_index]:
            new_position = position_value
        elif current_position == position_value and df[exit_column].iat[row_index]:
            new_position = 0

        df.iat[row_index, current_position_column] = current_position
        df.iat[row_index, new_position_column] = new_position

    return df


valid_directions = {"long", "short"}
invalid_directions = set(trade_directions) - valid_directions
if invalid_directions:
    raise ValueError(f"Invalid trade directions: {sorted(invalid_directions)}")
if hurdle_step <= 0:
    raise ValueError("hurdle_step must be greater than zero")

filename = f"{'_'.join(sorted_symbols_list)}_{moving_avg_days}.csv"
input_path = input_directory / filename
base_df = pd.read_csv(input_path, index_col=0)
base_df.columns = base_df.columns.str.strip()

base_df["from date"] = pd.to_datetime(base_df["from date"], errors="coerce")
base_df["to date"] = pd.to_datetime(base_df["to date"], errors="coerce")
date_mask = (
    base_df["from date"].between(start_date, end_date)
    & base_df["to date"].between(start_date, end_date)
)
base_df = base_df.loc[date_mask].reset_index(drop=True)

anchor = sorted_symbols_list[-1]
special_row = moving_avg_days - 1
special_column = f"to {anchor} / to {anchor}"
columns_to_clear = base_df.columns[
    base_df.columns.get_loc(special_column) + 1:
]
base_df.loc[:special_row, columns_to_clear] = np.nan

output_directory.mkdir(parents=True, exist_ok=True)

for direction in trade_directions:
    for entry_hurdle in build_hurdles(direction):
        exit_steps = int(round(abs(entry_hurdle) / hurdle_step))
        exit_sign = -1 if direction == "long" else 1
        exit_hurdles = [0.0] + [
            round(exit_sign * hurdle_step * i, 4)
            for i in range(1, exit_steps + 1)
        ]

        for exit_hurdle in exit_hurdles:
            df = add_positions(
                base_df, direction, entry_hurdle, exit_hurdle
            )
            hurdle_suffix = (
                f"_{direction}_{entry_hurdle:.4f}_{exit_hurdle:.4f}.csv"
            )
            output_path = output_directory / filename.replace(
                ".csv", hurdle_suffix
            )
            df.to_csv(output_path)
            print(f"Saved {output_path}")

print("finished")


Saved with positions\CWB_ICVT_20_long_0.0000_0.0000.csv
Saved with positions\CWB_ICVT_20_long_-0.0010_0.0000.csv
Saved with positions\CWB_ICVT_20_long_-0.0010_-0.0010.csv
Saved with positions\CWB_ICVT_20_long_-0.0020_0.0000.csv
Saved with positions\CWB_ICVT_20_long_-0.0020_-0.0010.csv
Saved with positions\CWB_ICVT_20_long_-0.0020_-0.0020.csv
Saved with positions\CWB_ICVT_20_long_-0.0030_0.0000.csv
Saved with positions\CWB_ICVT_20_long_-0.0030_-0.0010.csv
Saved with positions\CWB_ICVT_20_long_-0.0030_-0.0020.csv
Saved with positions\CWB_ICVT_20_long_-0.0030_-0.0030.csv
Saved with positions\CWB_ICVT_20_long_-0.0040_0.0000.csv
Saved with positions\CWB_ICVT_20_long_-0.0040_-0.0010.csv
Saved with positions\CWB_ICVT_20_long_-0.0040_-0.0020.csv
Saved with positions\CWB_ICVT_20_long_-0.0040_-0.0030.csv
Saved with positions\CWB_ICVT_20_long_-0.0040_-0.0040.csv
Saved with positions\CWB_ICVT_20_long_-0.0050_0.0000.csv
Saved with positions\CWB_ICVT_20_long_-0.0050_-0.0010.csv
Saved with positions\